*© 2026 Paul Fergus. Free for student and research use — commercial use is strictly prohibited.*

# Lab 9 — Zero-shot detection, and the build-vs-prompt decision

**Module:** Deep Learning Concepts and Techniques (Computer Vision)  
**Week:** 9  
**Estimated time:** 240 minutes

---

## ⚠ Requirements

This lab needs a **CUDA GPU**, your **trained model from Lab 7**, and the **ground-truth test set you built in Lab 8**. It reuses Lab 8's evaluation code, so keep that notebook handy.

> **Kernel hygiene reminder (see Lab 7).** If Lab 7 or Lab 8's kernel is still running in another tab, shut it down first — this lab loads a trained YOLO26 *and* YOLOE, and leftover models from earlier kernels eat into the VRAM headroom both need.

## Learning outcomes

By the end of this lab you should be able to:

1. Explain the difference between **closed-vocabulary** detection (a model trained on fixed classes) and **open-vocabulary / zero-shot** detection (a model that detects classes named at inference time via text prompts).
2. Run a modern zero-shot open-vocabulary detector (**YOLOE-26**) on your own images with no training and no annotation.
3. Evaluate a zero-shot model with the *same* rigorous metrics you built in Lab 8, and compare it head-to-head against your trained YOLO26.
4. Investigate how **prompt wording** affects zero-shot detection performance.
5. Apply a clear decision framework for *when to train your own model versus when to prompt a foundation model* — the central strategic judgement in applied computer vision today.
6. Use a foundation model as an **auto-labeller** to bootstrap a trained model — the hybrid approach that resolves the build-vs-prompt tension in practice.

## Prerequisites

- **Lab 7** — your trained `best.pt`.
- **Lab 8** — your ground-truth held-out set *and* the metric functions you wrote there.
- The textbook *Applied Deep Learning* (Fergus & Chalmers).
- Lecture 9: *Foundation models in vision — open-vocabulary detection and the shift from training to prompting*.

## The thread from last week, and the whole module

Over Labs 6, 7 and 8 you did the full closed-vocabulary pipeline: you **annotated** a dataset by hand, **trained** a YOLO26 detector on it, and **evaluated** it honestly on an independent ground-truth set. That is the classical, supervised way to build a detector, and it is still the right approach for many problems.

But the field has moved. Today there are **open-vocabulary** models — pretrained on enormous and diverse data — that detect objects you *name in plain text at inference time*, with no training and no labelled examples. So this lab asks the question that hangs over every modern computer-vision project:

> **Did you need to train anything at all? And if not always — when do you, and when don't you?**

We answer it empirically: run a zero-shot detector on your Lab 8 ground-truth set, score it with your Lab 8 metrics, and compare.

## Useful references

- [Ultralytics YOLOE documentation](https://docs.ultralytics.com/models/yoloe)
- [Ultralytics YOLO26 documentation](https://docs.ultralytics.com/models/yolo26)
- Cheng et al. (2024), *YOLO-World: Real-Time Open-Vocabulary Object Detection* — the work that brought open-vocabulary detection into the real-time YOLO family.

---

## 1. Closed-vocabulary vs open-vocabulary detection

<img src="assets/closed_vs_open_vocab.svg" alt="Closed vs open vocabulary detection" width="880"/>

**Closed-vocabulary** (what you built): the model has a *fixed* list of classes baked in at training time. Your YOLO26 knows exactly four things — buffalo, elephant, rhino, zebra — because those are what you annotated and trained on. Ask it about a giraffe and it cannot answer; the concept does not exist in its output layer. Getting there cost you hours of annotation (Lab 6) and GPU time (Lab 7).

**Open-vocabulary / zero-shot** (this lab): the model was pretrained on a huge, diverse corpus with a vision-language objective, so it has learned to associate *text* with *visual appearance* in general. At inference you hand it a list of class names as **text prompts** — `["buffalo", "elephant", ...]` — and it detects them, having never been trained on your specific task. 'Zero-shot' means zero training examples for your classes.

**YOLOE-26** is the model we'll use. It is the open-vocabulary member of the same YOLO26 family you trained in Lab 7, and it ships in the same `ultralytics` package — nothing new to install. It supports text prompts, visual prompts (give it an example image of the thing), and a prompt-free mode using a built-in vocabulary of 1200+ categories.

The trade-off is real and runs both ways, which is the whole point of this lab:

| | Trained YOLO26 (closed) | Zero-shot YOLOE-26 (open) |
|---|---|---|
| Data needed | hundreds of labelled boxes | none |
| Setup time | hours of annotation + training | seconds |
| Accuracy on *your* domain | usually higher | usually lower |
| New classes | retrain | just change the prompt |
| Runs offline / on edge | yes, small & fast | heavier model |
| Best when | narrow, fixed, high-volume task | flexible, exploratory, low-data task |

## 2. Setup — point at your Lab 7 model and Lab 8 ground truth

We reuse the *same* held-out set and the *same* evaluation machinery from Lab 8 so the comparison is fair: identical images, identical ground truth, identical metrics.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch

# Reuse your Lab 8 ground-truth set.
HOLDOUT_IMAGES = Path("../lab08_detection_evaluation/data/holdout_images")
HOLDOUT_LABELS = Path("../lab08_detection_evaluation/data/holdout_labels")

# Your trained model from Lab 7.
# Your trained model from Lab 7 (auto-discovered to survive run-name suffixes
# and Ultralytics' occasional path-nesting).
_lab7_root = Path("../lab07_yolo_training")
_candidates = sorted(_lab7_root.rglob("weights/best.pt"), key=lambda p: p.stat().st_mtime) if _lab7_root.exists() else []
TRAINED_MODEL = _candidates[-1].resolve() if _candidates else Path("../lab07_yolo_training/runs/african_wildlife/weights/best.pt")

# Class list — read from Lab 6's classes.txt so it always matches training order.
_classes_file = Path("../lab06_image_annotation/data/classes.txt")
CLASSES = [ln.strip() for ln in _classes_file.read_text().splitlines() if ln.strip()]
colours = ["#d6336c", "#f59f00", "#2b8a3e", "#1c7ed6"]

image_files = sorted(HOLDOUT_IMAGES.glob("*.jpg")) if HOLDOUT_IMAGES.is_dir() else []
print(f"Held-out images:        {len(image_files)}")
print(f"Trained model present:  {TRAINED_MODEL.is_file()}")
print(f"CUDA available:         {torch.cuda.is_available()}")

assert len(image_files) > 0, "Build your Lab 8 ground-truth set first."
assert torch.cuda.is_available(), "This lab needs a CUDA GPU (see Lab 7)."

### Bring across the Lab 8 evaluation functions

Rather than re-derive them, we re-define the core metric functions from Lab 8 (IoU, matching, AP). They're reproduced here so this notebook is self-contained, but they are exactly what you built and tested last week.

In [ ]:
def iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter == 0:
        return 0.0
    area_a = (a[2]-a[0]) * (a[3]-a[1]); area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter)


def match_one_image(gts, preds, iou_thr):
    preds_sorted = sorted(preds, key=lambda p: -p[0])
    matched = set(); out = []
    for score, box in preds_sorted:
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gts):
            if j in matched:
                continue
            v = iou_xyxy(box, g)
            if v > best_iou:
                best_iou, best_j = v, j
        if best_iou >= iou_thr and best_j >= 0:
            matched.add(best_j); out.append((score, 1))
        else:
            out.append((score, 0))
    return out, len(gts)


def load_ground_truth(stem):
    label_file = HOLDOUT_LABELS / f"{stem}.txt"
    img_file = HOLDOUT_IMAGES / f"{stem}.jpg"
    if not label_file.is_file():
        return []
    with Image.open(img_file) as im:
        W, H = im.size
    boxes = []
    for line in label_file.read_text().splitlines():
        if not line.strip():
            continue
        cls, cx, cy, w, h = line.split()
        cls = int(cls); cx, cy, w, h = map(float, (cx, cy, w, h))
        boxes.append((cls, [(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H]))
    return boxes


def map_from_predictions(predictions, ground_truth, iou_thr=0.5):
    """mean AP over classes for a {stem: [(cls,score,box)]} prediction dict."""
    aps = {}
    for cls_id in range(len(CLASSES)):
        scored = []; n_gt = 0
        for stem in ground_truth:
            cls_gts = [b for c, b in ground_truth[stem] if c == cls_id]
            cls_preds = [(s, b) for c, s, b in predictions.get(stem, []) if c == cls_id]
            m, ng = match_one_image(cls_gts, cls_preds, iou_thr)
            scored.extend(m); n_gt += ng
        if n_gt == 0:
            continue
        scored.sort(key=lambda p: -p[0])
        tp = np.array([t for _, t in scored]); fp = 1 - tp
        tpc, fpc = np.cumsum(tp), np.cumsum(fp)
        rec = tpc / n_gt; prec = tpc / np.maximum(tpc + fpc, 1e-9)
        mrec = np.concatenate([[0.0], rec, [1.0]]); mpre = np.concatenate([[0.0], prec, [0.0]])
        for i in range(len(mpre)-1, 0, -1):
            mpre[i-1] = max(mpre[i-1], mpre[i])
        idx = np.where(mrec[1:] != mrec[:-1])[0]
        aps[cls_id] = float(np.sum((mrec[idx+1]-mrec[idx]) * mpre[idx+1]))
    mean_ap = float(np.mean(list(aps.values()))) if aps else 0.0
    return mean_ap, aps


ground_truth = {p.stem: load_ground_truth(p.stem) for p in image_files}
print(f"Loaded ground truth for {len(ground_truth)} images, "
      f"{sum(len(v) for v in ground_truth.values())} boxes total.")

## 3. Baseline: re-score your trained YOLO26

First, reproduce your Lab 8 result — run your *trained* model on the held-out set and compute its mAP50. This is the number the zero-shot model must beat (or not).

In [ ]:
from ultralytics import YOLO

def collect_predictions(model, stems_files, conf=0.001):
    """Run a model over images, return {stem: [(cls,score,[x1,y1,x2,y2])]}."""
    preds = {}
    results = model.predict(source=[str(p) for p in stems_files],
                            conf=conf, device=0, verbose=False)
    for r, p in zip(results, stems_files):
        dets = []
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy()
            cf = r.boxes.conf.cpu().numpy()
            cl = r.boxes.cls.cpu().numpy().astype(int)
            for i in range(len(xyxy)):
                dets.append((int(cl[i]), float(cf[i]), xyxy[i].tolist()))
        preds[p.stem] = dets
    return preds


trained = YOLO(str(TRAINED_MODEL))
trained_preds = collect_predictions(trained, image_files)
trained_map50, trained_aps = map_from_predictions(trained_preds, ground_truth, 0.5)
print(f"Trained YOLO26 — mAP50 on held-out set: {trained_map50:.4f}")
for cid, ap in trained_aps.items():
    print(f"  {CLASSES[cid]:<10s} AP={ap:.4f}")

# `trained` has done its one job here (producing trained_preds, which is what
# the rest of the notebook actually compares against) — free it before
# Section 4 loads YOLOE, rather than leaving two detection models resident
# at once for no further benefit.
del trained
torch.cuda.empty_cache()

## 4. Zero-shot detection with YOLOE-26

Now the headline. We load YOLOE — a pretrained open-vocabulary model — give it our five class names **as text prompts**, and run it on the *same* images. **There is no training cell. That is the entire point.**

The key API is two lines:

```python
model.set_classes(names, model.get_text_pe(names))   # encode the text prompts
results = model.predict(image)                        # detect them
```

`get_text_pe` computes the *text positional embeddings* (the prompts run through the model's text encoder); `set_classes` installs them as the detection vocabulary. After that, `predict` behaves exactly like the trained model — same `Results` objects — so our Lab 8 metric code applies unchanged.

> **First run downloads the YOLOE weights** (a few hundred MB). On an offline machine, fetch them beforehand. If the exact checkpoint name below isn't available in your `ultralytics` version, check `https://docs.ultralytics.com/models/yoloe` for the current filename and edit the string — the API is identical.
>
> **YOLOE text prompts need CLIP** (the vision-language model that turns your words into embeddings). The container installs it at build time. If you ever see `ModuleNotFoundError: No module named 'clip'`, run the install cell immediately below, then restart the kernel.

In [ ]:
# RECOVERY CELL — only needed if you hit `ModuleNotFoundError: No module named 'clip'`.
# The container should already have CLIP, so normally you can skip this. If you do
# need it, run this cell, then restart the kernel (Kernel -> Restart) and continue.
import importlib.util
if importlib.util.find_spec("clip") is None:
    import sys, subprocess
    print("CLIP not found — installing clip-anytorch ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--user",
                    "clip-anytorch", "ftfy", "regex"], check=False)
    print("Done. Now restart the kernel (Kernel -> Restart) and re-run from the top.")
else:
    print("CLIP is available — no action needed.")

In [ ]:
from ultralytics import YOLOE

# Load a pretrained open-vocabulary YOLOE model.
# yoloe-11l-seg.pt is a widely-available checkpoint; a yoloe-26 variant will
# work identically if present in your ultralytics version.
yoloe = YOLOE("yoloe-11l-seg.pt")

# Set our classes as TEXT PROMPTS — no training involved.
prompt_names = list(CLASSES)
yoloe.set_classes(prompt_names, yoloe.get_text_pe(prompt_names))

print("YOLOE loaded and prompted with:", prompt_names)
print("Model's active class names:", yoloe.names)

In [ ]:
# Run zero-shot inference on the SAME held-out images.
# YOLOE's prompt index order matches prompt_names, which we deliberately set
# to the same order as CLASSES — so class ids line up with the ground truth.
zeroshot_preds = collect_predictions(yoloe, image_files, conf=0.001)
zeroshot_map50, zeroshot_aps = map_from_predictions(zeroshot_preds, ground_truth, 0.5)

print(f"Zero-shot YOLOE — mAP50 on held-out set: {zeroshot_map50:.4f}")
for cid, ap in zeroshot_aps.items():
    print(f"  {CLASSES[cid]:<10s} AP={ap:.4f}")

## 5. Head-to-head

The same images, the same ground truth, the same metric — one model trained for hours on hand-labelled data, the other given five words and zero training. Let's see how they compare.

In [ ]:
print(f"{'Class':<12s}{'Trained AP':>12s}{'Zero-shot AP':>14s}")
print("-" * 38)
for cid in range(len(CLASSES)):
    t = trained_aps.get(cid, 0.0); z = zeroshot_aps.get(cid, 0.0)
    print(f"{CLASSES[cid]:<12s}{t:>12.3f}{z:>14.3f}")
print("-" * 38)
print(f"{'mAP50':<12s}{trained_map50:>12.3f}{zeroshot_map50:>14.3f}")

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(CLASSES)); w = 0.38
ax.bar(x - w/2, [trained_aps.get(i, 0) for i in range(len(CLASSES))], w,
       label="Trained YOLO26", color="#15161a")
ax.bar(x + w/2, [zeroshot_aps.get(i, 0) for i in range(len(CLASSES))], w,
       label="Zero-shot YOLOE", color="#c0322b")
ax.set_xticks(x); ax.set_xticklabels(CLASSES)
ax.set_ylabel("AP@0.50"); ax.set_ylim(0, 1)
ax.set_title("Trained vs zero-shot — per-class AP on your held-out set")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

**How to read this.** Don't expect a clean win for either side. Typically:

- The **trained** model wins on classes where your training data was representative — it has specialised to your exact domain.
- The **zero-shot** model can surprise you, especially on visually-distinctive, common animals (elephants, zebras) that are well-represented in its huge pretraining corpus — *with no data work from you at all*.
- Zero-shot often produces more false positives (it's eager) and looser boxes (lower mAP at strict IoU).

Whatever your specific numbers, the lesson is the same: **a zero-shot model gets you a non-trivial detector for free**, which reframes when the weeks of annotation-and-training are actually worth it.

### See the difference

Render the two models' predictions side by side on a few images.

In [ ]:
def draw_preds(ax, stem, preds, title):
    ax.imshow(Image.open(HOLDOUT_IMAGES / f"{stem}.jpg")); ax.axis("off")
    ax.set_title(title, fontsize=10)
    for cls_id, score, b in preds.get(stem, []):
        if score < 0.25:
            continue
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                     linewidth=2, edgecolor=colours[cls_id % len(colours)], facecolor="none"))
        ax.text(b[0], b[1]-3, f"{CLASSES[cls_id]} {score:.2f}", color="white", fontsize=8,
                bbox=dict(boxstyle="square,pad=0.1", facecolor=colours[cls_id % len(colours)], edgecolor="none"))

n_show = min(3, len(image_files))
fig, axes = plt.subplots(n_show, 2, figsize=(12, 4.5 * n_show))
if n_show == 1:
    axes = axes.reshape(1, 2)
for row, p in enumerate(image_files[:n_show]):
    draw_preds(axes[row][0], p.stem, trained_preds, f"{p.stem} — trained YOLO26")
    draw_preds(axes[row][1], p.stem, zeroshot_preds, f"{p.stem} — zero-shot YOLOE")
plt.tight_layout(); plt.show()

## 6. Prompt engineering for detection

Zero-shot performance depends on the *words* you choose. The model maps your prompt into a shared vision-language space, so a more descriptive or more canonical prompt can match the visual concept better. This is a genuinely new skill: with a trained model you tune weights; with a foundation model you tune *prompts*.

Below we try richer prompts for one class and see whether mAP for that class moves.

> **Implementation note.** Re-calling `set_classes` repeatedly on one model instance can raise an "inference tensors" error in some `ultralytics` versions. To stay safe we reload a fresh YOLOE for each prompt set. It's a little slower but robust.

In [ ]:
def zeroshot_map_with_prompts(prompt_list):
    """prompt_list maps each class in CLASSES to a prompt string (same order as CLASSES).
    Returns (mAP50, per-class AP). Reloads YOLOE fresh to avoid set_classes reuse issues."""
    m = YOLOE("yoloe-11l-seg.pt")
    m.set_classes(prompt_list, m.get_text_pe(prompt_list))
    preds = collect_predictions(m, image_files, conf=0.001)
    result = map_from_predictions(preds, ground_truth, 0.5)
    # Each call loads its own fresh YOLOE instance — free it before returning
    # rather than leaving it for the garbage collector to get to eventually.
    del m
    torch.cuda.empty_cache()
    return result


# Baseline prompts (plain class names) vs richer, more descriptive phrasing.
plain = list(CLASSES)
rich  = [f"a wild {c}" for c in CLASSES]

plain_map, _ = zeroshot_map_with_prompts(plain)
rich_map, _ = zeroshot_map_with_prompts(rich)

print(f"Plain prompts  {plain}  -> mAP50 {plain_map:.4f}")
print(f"Rich prompts   {rich}  -> mAP50 {rich_map:.4f}")
print(f"\nDifference: {rich_map - plain_map:+.4f}")
print("Richer, more descriptive prompts sometimes help and sometimes hurt —")
print("the model's pretraining vocabulary decides. This is prompt engineering.")

## 7. The decision framework — train or prompt?

You now have empirical data for *your* task. Here is the framework practitioners use to decide, with the questions that matter:

**Lean towards training your own (closed-vocabulary) when:**
- The task is **narrow and fixed** — a known, stable set of classes you'll detect for a long time.
- You need **maximum accuracy on a specific domain** (your exact cameras, lighting, species).
- You need a **small, fast model** for edge deployment or high throughput.
- You must **run fully offline** with a compact model.
- You *have* (or can afford to create) labelled data.

**Lean towards prompting a foundation model (open-vocabulary) when:**
- You have **little or no labelled data** and limited time.
- The set of classes is **fluid or exploratory** — you don't yet know what you'll need to detect.
- You're building a **prototype** and want a result today.
- You need to detect a **long tail** of rare classes that you'll never have enough data to train on.

**The hybrid — and this is what most mature teams actually do:**
- Use a foundation model to **auto-label** a first dataset, have humans correct it (far faster than labelling from scratch), then **train** a small specialised model on the corrected labels. You get the foundation model's breadth *and* the trained model's speed and domain accuracy. We try this in Exercise 3.

There is no universal right answer — the skill is diagnosing which situation you're in. Most of the failures in industry come from training a bespoke model when a prompt would have done, or shipping a zero-shot prototype when the problem demanded a specialised model.

## 8. Exercise 1 — push the zero-shot model

Investigate how far you can take the zero-shot model *without training*.

**(a)** Try at least **four** different prompt phrasings for your weakest class (the one with the lowest zero-shot AP in Section 5). Examples: bare name, species name, name + colour, name + a descriptive phrase (e.g. 'a large grey animal with a horn' for rhino). Record mAP for that class under each prompt. Which wording wins, and can you explain why in terms of what the model was pretrained on?

**(b)** Sweep the confidence threshold for the zero-shot model (reuse your Lab 8 threshold-sweep code) and plot precision-recall. Does the zero-shot model need a *different* operating threshold than your trained model did? Why might that be?

**(c)** Try adding a class the model was **never trained on by you and isn't in your ground truth** — e.g. add `"giraffe"` to the prompt list — and look at what it detects. This demonstrates the open-vocabulary superpower: detecting things you never planned for. Comment on what you see.

In [ ]:
# Your code for Exercise 1 here.


*Your findings:*



## 9. Exercise 2 — when does each model win?

Using the side-by-side renders (Section 5) and your metrics, find concrete examples of:

**(a)** An image where the **trained** model clearly beats zero-shot. Explain *why* — what about this image plays to the trained model's domain specialisation?

**(b)** An image where the **zero-shot** model does as well or better. What made it easy for a general-purpose model?

**(c)** Based on your whole analysis, write a short recommendation (5–8 sentences) for a *specific* hypothetical client: an African wildlife conservation charity that wants to monitor **20 species** (not 4) across **hundreds of trail cameras**, has a **small budget**, and **no existing labelled data**. Train, prompt, or hybrid? Justify using evidence from this lab.

In [ ]:
# Your code / analysis for Exercise 2 here.


*Your recommendation:*



## 10. Exercise 3 — the hybrid: auto-label, then train

This is how the build-vs-prompt tension actually resolves in industry, and it brings the whole module full circle.

1. Take a folder of **new, unlabelled** wildlife images (source 15–20 more, or reuse spare Lab 8 images you didn't annotate).
2. Run the **zero-shot YOLOE** model on them and save its predictions **as YOLO-format label files** (the auto-labels). Ultralytics can write these with `predict(..., save_txt=True)`, or write them yourself from the boxes.
3. **Inspect and correct** a few in the Lab 6 annotator — note how much faster correcting machine labels is than drawing from scratch.
4. Add these auto-labelled (and corrected) images to your Lab 6/7 training set and **fine-tune** your YOLO26 for a few epochs.
5. Re-evaluate on your Lab 8 held-out set using `map_from_predictions`. **Did the extra auto-labelled data improve your trained model's mAP?**

Write up what you found. Even a small change is informative: it tells you whether foundation-model auto-labelling is a viable shortcut to more training data for this task. This is the measure → intervene → re-measure loop from the scientific method, applied to your own model.

In [ ]:
# Your code for Exercise 3 here.


*Did auto-labelling + fine-tuning improve your model? What does that tell you?*



---

## 11. Reflection questions

**Q1.** In your own words, what does 'zero-shot' mean, and how can a model detect a class it was never explicitly trained on by you?

**Q2.** Give two concrete situations from outside wildlife monitoring — one where you'd train a closed-vocabulary model, one where you'd prompt an open-vocabulary model — and justify each choice.

**Q3.** Zero-shot models often produce more false positives than a specialised trained model. Why might a general-purpose, eager-to-detect model behave this way, and how does the confidence threshold help you manage it?

**Q4.** Prompt engineering changed your zero-shot results. What does the sensitivity of detection to prompt wording tell you about how these models represent visual concepts?

**Q5.** Reflecting across the whole module — from the perceptron in Lab 1 to foundation models here — write a few sentences on how the *role of the practitioner* has changed as the tools have grown more powerful. What stays essential no matter how good the models get? (Hint: think about Lab 8.)

*Your answers:*

**A1.** 

**A2.** 

**A3.** 

**A4.** 

**A5.** 

---

## Onward — this isn't the end of the module

Over nine labs you have gone from implementing a perceptron by hand to training, honestly evaluating, and critically comparing a state-of-the-art object detector against a foundation model. You can now:

- build and train neural networks and CNNs from first principles;
- handle real image data and annotate it for detection;
- train and fine-tune modern detectors on your own data;
- **evaluate honestly** — the skill that separates practitioners from button-pushers;
- and reason about *when to build versus when to prompt* in the foundation-model era.

The single most durable lesson is the one from Lab 8, reinforced here: **the models will keep changing — what stays essential is the discipline of measuring honestly and choosing the right tool for the actual problem.** That discipline is what makes you valuable long after today's state of the art is obsolete.

**Where to go next:** two more labs pick up threads left dangling above. **Lab 10** turns the boxes you've been detecting into pixel-accurate masks — instance segmentation, using a promptable foundation model (SAM) the same way you used YOLOE here, to label data with no manual tracing. **Lab 11** then turns the question around entirely: instead of asking what a model predicts, it asks *why* — opening up your trained CNNs with Grad-CAM and saliency maps to check whether they're using genuine evidence or a shortcut.

Before you finish:

- [ ] You ran both the trained and zero-shot models on your held-out set and compared them
- [ ] You completed Exercises 1, 2, and 3
- [ ] You answered the reflection questions
- [ ] Your notebook runs top to bottom without errors